In [1]:
import pandas as pd
import geopandas as gpd
from shapely import wkt


In [3]:
# -----------------------------
# 1. Community Areas
# -----------------------------
community_df = pd.read_csv("/home/metayj/InternetGIS/CrimeAnalysis/data/raw/Boundaries_-_Community_Areas_20260317.csv")
community_df["geometry"] = community_df["the_geom"].apply(wkt.loads)
community_gdf = gpd.GeoDataFrame(community_df, geometry="geometry", crs="EPSG:4326")
community_gdf = community_gdf[["AREA_NUMBE", "COMMUNITY", "SHAPE_AREA", "SHAPE_LEN", "geometry"]]

print(community_gdf.head())
print(community_gdf.crs)
print(community_gdf.geom_type.unique())

   AREA_NUMBE       COMMUNITY       SHAPE_AREA       SHAPE_LEN  \
0           1     ROGERS PARK  51,259,902.4506  34,052.3975757   
1           2      WEST RIDGE  98,429,094.8621  43,020.6894583   
2           3          UPTOWN  65,095,642.7289  46,972.7945549   
3           4  LINCOLN SQUARE  71,352,328.2399  36,624.6030848   
4           5    NORTH CENTER    57,054,167.85  31,391.6697542   

                                            geometry  
0  MULTIPOLYGON (((-87.65456 41.99817, -87.65574 ...  
1  MULTIPOLYGON (((-87.68465 42.01948, -87.68464 ...  
2  MULTIPOLYGON (((-87.64102 41.9548, -87.644 41....  
3  MULTIPOLYGON (((-87.67441 41.9761, -87.6744 41...  
4  MULTIPOLYGON (((-87.67336 41.93234, -87.67342 ...  
EPSG:4326
<StringArray>
['MultiPolygon']
Length: 1, dtype: str


In [4]:
community_gdf.to_file("/home/metayj/InternetGIS/CrimeAnalysis/data/processed/community_areas.geojson", driver="GeoJSON")

In [5]:
# -----------------------------
# 2. CTA Stations
# -----------------------------
cta_df = pd.read_csv("/home/metayj/InternetGIS/CrimeAnalysis/data/raw/CTA_-_'L'_(Rail)_Stations_20260317.csv")
cta_df["geometry"] = cta_df["the_geom"].apply(wkt.loads)
cta_gdf = gpd.GeoDataFrame(cta_df, geometry="geometry", crs="EPSG:4326")
cta_gdf = cta_gdf[["STATION_ID", "LONGNAME", "LINES", "ADA", "LEGEND", "geometry"]]

print(cta_gdf.head())
print(cta_gdf.geom_type.unique())

   STATION_ID           LONGNAME  \
0         970    Cicero-Congress   
1          20        Harlem-Lake   
2         610          Ridgeland   
3         230         Cumberland   
4        1700  Washington/Wabash   

                                           LINES    ADA          LEGEND  \
0                           Blue Line (Congress)  False       Blue Line   
1                              Green Line (Lake)   True      Green Line   
2                              Green Line (Lake)  False      Green Line   
3                                      Blue Line   True       Blue Line   
4  Brown, Orange, Pink,  Purple (Express), Green   True  Multiple Lines   

                     geometry  
0  POINT (-87.74517 41.87161)  
1  POINT (-87.80318 41.88685)  
2  POINT (-87.78366 41.88716)  
3  POINT (-87.83803 41.98429)  
4  POINT (-87.62619 41.88322)  
<StringArray>
['Point']
Length: 1, dtype: str


In [6]:
cta_gdf.to_file("/home/metayj/InternetGIS/CrimeAnalysis/data/processed/cta_stations.geojson", driver="GeoJSON")

In [7]:
# -----------------------------
# 3. Police Stations
# -----------------------------
police_df = pd.read_csv("/home/metayj/InternetGIS/CrimeAnalysis/data/raw/Police_Stations_20260317.csv")
police_df = police_df.dropna(subset=["LATITUDE", "LONGITUDE"]).copy()
police_gdf = gpd.GeoDataFrame(
    police_df,
    geometry=gpd.points_from_xy(police_df["LONGITUDE"], police_df["LATITUDE"]),
    crs="EPSG:4326"
)
police_gdf = police_gdf[["DISTRICT", "DISTRICT NAME", "ADDRESS", "LATITUDE", "LONGITUDE", "geometry"]]

print(police_gdf.head())
print(police_gdf.geom_type.unique())

       DISTRICT DISTRICT NAME              ADDRESS   LATITUDE  LONGITUDE  \
0  Headquarters  Headquarters  3510 S Michigan Ave  41.830702 -87.623395   
1            18    Near North   1160 N Larrabee St  41.903242 -87.643352   
2            19     Town Hall     850 W Addison St  41.947400 -87.651512   
3            20       Lincoln   5400 N Lincoln Ave  41.979550 -87.692845   
4            22   Morgan Park  1900 W Monterey Ave  41.691435 -87.668520   

                     geometry  
0    POINT (-87.6234 41.8307)  
1  POINT (-87.64335 41.90324)  
2   POINT (-87.65151 41.9474)  
3  POINT (-87.69284 41.97955)  
4  POINT (-87.66852 41.69143)  
<StringArray>
['Point']
Length: 1, dtype: str


In [9]:
police_gdf.to_file("/home/metayj/InternetGIS/CrimeAnalysis/data/processed/police_stations.geojson", driver="GeoJSON")

In [10]:
# -----------------------------
# 4. Crimes (filter to 2025)
# -----------------------------
crime_df = pd.read_csv("/home/metayj/InternetGIS/CrimeAnalysis/data/raw/Crimes_-_2001_to_Present_20260317.csv", low_memory=False)
crime_df["Date"] = pd.to_datetime(crime_df["Date"], errors="coerce")
crime_df = crime_df[
    (crime_df["Year"] == 2025) &
    crime_df["Latitude"].notna() &
    crime_df["Longitude"].notna()
].copy()

crime_keep = [
    "ID", "Date", "Primary Type", "Description", "Location Description",
    "Arrest", "Domestic", "District", "Ward", "Community Area",
    "Year", "Latitude", "Longitude"
]
crime_df = crime_df[crime_keep]

crime_gdf = gpd.GeoDataFrame(
    crime_df,
    geometry=gpd.points_from_xy(crime_df["Longitude"], crime_df["Latitude"]),
    crs="EPSG:4326"
)

print(crime_gdf.head())
print(len(crime_gdf))

/tmp/ipykernel_7469/570412550.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  crime_df["Date"] = pd.to_datetime(crime_df["Date"], errors="coerce")


             ID                Date            Primary Type  \
37108  14075483 2025-12-31 23:58:00                 ASSAULT   
37109  14070833 2025-12-31 23:55:00     MOTOR VEHICLE THEFT   
37110  14070745 2025-12-31 23:54:00  PUBLIC PEACE VIOLATION   
37111  14070845 2025-12-31 23:54:00                 BATTERY   
37112  14070799 2025-12-31 23:54:00  PUBLIC PEACE VIOLATION   

                                             Description Location Description  \
37108                                             SIMPLE            RESIDENCE   
37109                      THEFT / RECOVERY - AUTOMOBILE            APARTMENT   
37110                                    OTHER VIOLATION             AIRCRAFT   
37111  AGGRAVATED P.O. - HANDS, FISTS, FEET, NO / MIN...           RESTAURANT   
37112                                    OTHER VIOLATION             AIRCRAFT   

       Arrest  Domestic  District  Ward  Community Area  Year   Latitude  \
37108   False     False       9.0  20.0            61.0  2

In [11]:
crime_gdf.to_file("/home/metayj/InternetGIS/CrimeAnalysis/data/processed/crime_points_2025.geojson", driver="GeoJSON")

In [12]:
# -----------------------------
# 5. Business Licenses
# -----------------------------
biz_df = pd.read_csv("/home/metayj/InternetGIS/CrimeAnalysis/data/raw/Business_Licenses_20260317.csv", low_memory=False)
biz_df = biz_df.dropna(subset=["LATITUDE", "LONGITUDE"]).copy()

biz_keep = [
    "LICENSE ID", "LEGAL NAME", "DOING BUSINESS AS NAME",
    "LICENSE DESCRIPTION", "BUSINESS ACTIVITY",
    "LICENSE STATUS", "COMMUNITY AREA", "LATITUDE", "LONGITUDE"
]
biz_df = biz_df[biz_keep]

biz_gdf = gpd.GeoDataFrame(
    biz_df,
    geometry=gpd.points_from_xy(biz_df["LONGITUDE"], biz_df["LATITUDE"]),
    crs="EPSG:4326"
)

print(biz_gdf.head())
print(len(biz_gdf))

   LICENSE ID                             LEGAL NAME  \
0     3074876  ACCESS LIVING OF METROPOLITAN CHICAGO   
1     3077224                 48forty Solutions, LLC   
2     3077736                   JOLIE FLEUR, LIMITED   
3     3075206     GREAT LAKES FINANCIAL SERVICES INC   
4     3017968                SANCHEZ BIKE SHOP, INC.   

                   DOING BUSINESS AS NAME           LICENSE DESCRIPTION  \
0  ACCESS LIVING OF METROPOLITAN  CHICAGO                       Raffles   
1                  48forty Solutions, LLC  Manufacturing Establishments   
2                     JOLIE FLEUR LIMITED      Limited Business License   
3    GREAT LAKES FINANCIAL SERVICES, INC.      Limited Business License   
4                       SANCHEZ BIKE SHOP             Secondhand Dealer   

                                   BUSINESS ACTIVITY LICENSE STATUS  \
0  Not-For-Profit Selling Raffles for Prizes of $...            AAI   
1               Manufacturing of Miscellaneous Items            AAI   

In [13]:
biz_gdf.to_file("/home/metayj/InternetGIS/CrimeAnalysis/data/processed/business_licenses_clean.geojson", driver="GeoJSON")

In [14]:
# -----------------------------
# 6. Crimes (filter to 2015)
# -----------------------------
crime_df = pd.read_csv("/home/metayj/InternetGIS/CrimeAnalysis/data/raw/Crimes_-_2001_to_Present_20260317.csv", low_memory=False)
crime_df["Date"] = pd.to_datetime(crime_df["Date"], errors="coerce")
crime_df = crime_df[
    (crime_df["Year"] == 2015) &
    crime_df["Latitude"].notna() &
    crime_df["Longitude"].notna()
].copy()

crime_keep = [
    "ID", "Date", "Primary Type", "Description", "Location Description",
    "Arrest", "Domestic", "District", "Ward", "Community Area",
    "Year", "Latitude", "Longitude"
]
crime_df = crime_df[crime_keep]

crime_gdf = gpd.GeoDataFrame(
    crime_df,
    geometry=gpd.points_from_xy(crime_df["Longitude"], crime_df["Latitude"]),
    crs="EPSG:4326"
)

# print(crime_gdf.head())
print(len(crime_gdf))

crime_gdf.to_file("/home/metayj/InternetGIS/CrimeAnalysis/data/processed/crime_points_2015.geojson", driver="GeoJSON")
print("done")

/tmp/ipykernel_7469/2295030353.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  crime_df["Date"] = pd.to_datetime(crime_df["Date"], errors="coerce")


257908
done


/home/metayj/InternetGIS/CrimeAnalysis/